# Stage 2: Model Training

DVC stage `train`. Trains a Random Forest classifier that predicts whether a booking will be cancelled (`is_canceled`). Hyperparameters come from the `train` section of `params.yaml`.

In [ ]:
import json
import time
from pathlib import Path

import joblib
import pandas as pd
import yaml
from sklearn.ensemble import RandomForestClassifier

## Load parameters and training data

In [ ]:
with open('params.yaml') as f:
    params = yaml.safe_load(f)

seed = params['base']['random_state']
train_cfg = params['train']
target = params['base']['target']

train_df = pd.read_csv(Path(params['data']['processed_dir']) / 'train.csv')
X_train = train_df.drop(columns=[target])
y_train = train_df[target]
print(f'X_train: {X_train.shape}')
train_cfg

## Train the model

In [ ]:
model = RandomForestClassifier(
    n_estimators=train_cfg['n_estimators'],
    max_depth=train_cfg['max_depth'],
    min_samples_split=train_cfg['min_samples_split'],
    min_samples_leaf=train_cfg['min_samples_leaf'],
    class_weight=train_cfg['class_weight'],
    n_jobs=-1,
    random_state=seed,
)

start = time.time()
model.fit(X_train, y_train)
print(f'Training time: {time.time() - start:.1f}s')
print(f'Training accuracy: {model.score(X_train, y_train):.4f}')

## Save the model

In [ ]:
Path('models').mkdir(exist_ok=True)
joblib.dump({'model': model, 'features': X_train.columns.tolist()}, 'models/model.pkl', compress=3)
print('Saved models/model.pkl')